## 3、工具的定义方式2：使用 @tool 装饰器

对应《尚硅谷-05-Tools.pdf》第 3 节（PDF 第 14–26 页）。

保持课件顺序：`description → parse_docstring → name_or_callable → args_schema`。
下面全部是离线示例，天气数据为模拟数据，不需要模型 API；选择 `langchain` 内核。
各小节会重新定义 `get_weather`，用于比较不同写法，按顺序运行即可。

In [1]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

### 3.1 自定义工具描述：description

#### 3.1.1 只有 @tool，却没有 docstring

没有显式 `description` 时，需要 docstring。这里在装饰函数时就会报错。

In [2]:
try:
    @tool
    def get_weather(city: str) -> str:
        return f"{city}天气晴朗~"
except ValueError as e:
    print(type(e).__name__, e)

ValueError Function must have a docstring if description not provided.


##### 补充 docstring

一个简短的描述就能定义工具；只有需要解析参数说明时，才需要相应的 Google 风格结构。

In [3]:
@tool
def get_weather(city: str) -> str:
    """天气查询工具。"""
    return f"{city}天气晴朗~"

print(type(get_weather).__name__)
print(get_weather.name)
rprint(convert_to_openai_tool(get_weather))
print(get_weather.invoke({"city": "北京"}))

StructuredTool
get_weather


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具。',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

北京天气晴朗~


#### 3.1.2 显式 description：优先级高于 docstring 的函数说明

In [4]:
@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str) -> str:
    """天气查询工具。"""
    return f"{city}天气晴朗~"

print(get_weather.description)
rprint(convert_to_openai_tool(get_weather))

根据城市名称查询当日天气的工具


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

#### 3.1.3 parse_docstring：是否解析 Args 中的参数说明

默认 `parse_docstring=False`：整个 docstring 被当作工具描述；
`Args:` 不会自动拆成各参数的 `description`。

In [5]:
@tool
def get_weather(
    city: str,
    units: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """获取当日天气，可选择是否同时查询未来五日天气预报。

    Args:
        city: 城市名称。
        units: 气温单位，celsius 为摄氏度，fahrenheit 为华氏度。
        include_forecast: 是否包含未来五日天气预报。
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当日气温：{temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

without_parse_schema = convert_to_openai_tool(get_weather)
rprint(without_parse_schema)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报。\n\nArgs:\n    city: 城市名称。\n    
units: 气温单位，celsius 为摄氏度，fahrenheit 为华氏度。\n    include_forecast: 是否包含未来五日天气预报。',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'units': {'default': 'celsius', 'type': 'string'},
                'include_forecast': {'default': False, 'type': 'boolean'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

##### 设置 parse_docstring=True，再观察参数 description

函数体不变，只改装饰器；注意摘要和 `Args:` 中间的空行。

In [6]:
@tool(parse_docstring=True)
def get_weather(
    city: str,
    units: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """获取当日天气，可选择是否同时查询未来五日天气预报。

    Args:
        city: 城市名称。
        units: 气温单位，celsius 为摄氏度，fahrenheit 为华氏度。
        include_forecast: 是否包含未来五日天气预报。
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当日气温：{temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

with_parse_schema = convert_to_openai_tool(get_weather)
rprint(with_parse_schema)
print(get_weather.invoke({"city": "北京", "include_forecast": True}))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报。',
        'parameters': {
            'properties': {
                'city': {'description': '城市名称。', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，celsius 为摄氏度，fahrenheit 为华氏度。',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日天气预报。',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

北京当日气温：22 摄氏度
未来五天都是晴天


In [7]:
before = without_parse_schema["function"]
after = with_parse_schema["function"]

print("解析前工具描述：", before["description"])
print("解析后工具描述：", after["description"])
print("解析前 city 描述：", before["parameters"]["properties"]["city"].get("description"))
print("解析后 city 描述：", after["parameters"]["properties"]["city"].get("description"))

解析前工具描述： 获取当日天气，可选择是否同时查询未来五日天气预报。

Args:
    city: 城市名称。
    units: 气温单位，celsius 为摄氏度，fahrenheit 为华氏度。
    include_forecast: 是否包含未来五日天气预报。
解析后工具描述： 获取当日天气，可选择是否同时查询未来五日天气预报。
解析前 city 描述： None
解析后 city 描述： 城市名称。


##### 易错：启用解析后，docstring 格式不合法

例如摘要与 `Args:` 之间缺少空行，默认会抛出 `ValueError`。
这与「只写一个简短 docstring、不解析参数」是两种情况。

In [8]:
try:
    @tool(parse_docstring=True)
    def get_weather(city: str) -> str:
        """天气查询工具。
        Args:
            city: 城市名称。
        """
        return f"{city}天气晴朗~"
except ValueError as e:
    print(type(e).__name__, e)

ValueError Found invalid Google-Style docstring.


### 3.2 更改工具名称：name_or_callable

默认名称来自函数名；`@tool(name_or_callable="getWeather")` 可以指定模型看到的工具名。
也可以简写为 `@tool("getWeather")`。

In [9]:
@tool(name_or_callable="getWeather")
def get_weather(city: str) -> str:
    """天气查询工具。"""
    return f"{city}天气晴朗~"

print("Python 中的变量名：get_weather")
print("模型看到的工具名：", get_weather.name)
rprint(convert_to_openai_tool(get_weather))
print(get_weather.invoke({"city": "上海"}))

Python 中的变量名：get_weather
模型看到的工具名： getWeather


{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具。',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

上海天气晴朗~


通常直接使用函数名即可。若改名，分发工具时要匹配 `tool_call["name"]` 中的新名称。
普通业务参数避免使用 `config`、`runtime` 这些框架保留名称。

### 3.3 自定义 args_schema

#### 3.3.1 方式1：使用 Pydantic 模型定义

##### ① BaseModel：定义参数结构

继承 `BaseModel`，用类型注解声明字段；实例化时按关键字传入数据。

In [10]:
class WeatherInput(BaseModel):
    city: str

weather_input = WeatherInput(city="北京")
print(weather_input)
print(weather_input.model_dump())
rprint(WeatherInput.model_json_schema())

city='北京'
{'city': '北京'}


{
    'properties': {'city': {'title': 'City', 'type': 'string'}},
    'required': ['city'],
    'title': 'WeatherInput',
    'type': 'object'
}

##### 易错：BaseModel 不能按下面这样传入位置参数

In [11]:
try:
    WeatherInput("北京")
except TypeError as e:
    print(type(e).__name__, e)

TypeError BaseModel.__init__() takes 1 positional argument but 2 were given


##### ② Field：给字段添加默认值和描述

`Field(default=..., description=...)` 同时提供默认值和给模型看的参数含义。

In [12]:
class WeatherInput(BaseModel):
    city: str = Field(default="北京", description="城市名称")
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报",
    )

print(WeatherInput())
rprint(WeatherInput.model_json_schema())

city='北京' include_forecast=False


{
    'properties': {
        'city': {'default': '北京', 'description': '城市名称', 'title': 'City', 'type': 'string'},
        'include_forecast': {
            'default': False,
            'description': '是否包含未来五日天气预报',
            'title': 'Include Forecast',
            'type': 'boolean'
        }
    },
    'title': 'WeatherInput',
    'type': 'object'
}

##### ③ Literal：限定参数只能取几个固定值

普通 `str` 可以是任意字符串；`Literal` 可以把单位限制为摄氏或华氏。

In [13]:
class WeatherInput(BaseModel):
    city: str
    unit: Literal["celsius", "fahrenheit"]

print("合法：", WeatherInput(city="北京", unit="celsius"))

try:
    WeatherInput(city="北京", unit="kelvin")
except ValidationError as e:
    print("非法：", type(e).__name__)
    print(e)

合法： city='北京' unit='celsius'
非法： ValidationError
1 validation error for WeatherInput
unit
  Input should be 'celsius' or 'fahrenheit' [type=literal_error, input_value='kelvin', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error


##### ④ 把 Pydantic 参数模型绑定到工具

`@tool(args_schema=WeatherInput)` 同时控制参数描述和工具执行前的校验。
这里沿用课件的 `unit`（单数）；上面 docstring 示例中的 `units` 是另一个函数版本。
同一个示例中的名称必须始终一致。

In [14]:
class WeatherInput(BaseModel):
    city: str = Field(default="北京", description="城市名称")
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位",
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报",
    )

@tool(args_schema=WeatherInput)
def get_weather(
    city: str,
    unit: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """获取当日天气，可选未来五日天气预报。"""
    temp = 22 if unit == "celsius" else 72
    result = f'{city}当日气温：{temp} {"摄氏度" if unit == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选未来五日天气预报。',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '城市名称', 'type': 'string'},
                'unit': {
                    'default': 'celsius',
                    'description': '气温单位',
                    'enum': ['celsius', 'fahrenheit'],
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日天气预报',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }
    }
}

##### 先直接执行工具，检查默认值和不同输入

当前工程的 `langchain-core==1.2.18` 会把 Pydantic Schema 中的显式默认值传给工具。
所以本例即使函数签名中的 `city` 没有默认值，空参数字典也能使用 Schema 的「北京」。

In [15]:
print("全部使用默认值：")
print(get_weather.invoke({}))

print("指定城市、华氏度和预报：")
print(get_weather.invoke({
    "city": "上海",
    "unit": "fahrenheit",
    "include_forecast": True,
}))

全部使用默认值：
北京当日气温：22 摄氏度
指定城市、华氏度和预报：
上海当日气温：72 华氏度
未来五天都是晴天


##### 非法参数：Pydantic 在执行函数前拒绝输入

In [16]:
try:
    get_weather.invoke({"city": "北京", "unit": "kelvin"})
except ValidationError as e:
    print(type(e).__name__)
    print(e)

# 留作下面与 JSON Schema 字典的比较
pydantic_weather = get_weather

ValidationError
1 validation error for WeatherInput
unit
  Input should be 'celsius' or 'fahrenheit' [type=literal_error, input_value='kelvin', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error


#### 3.3.2 方式2：使用 JSON Schema 字典

传给 `args_schema` 的是参数结构字典，即 `parameters` 对应的内容，
**不是整份 `{"type": "function", "function": ...}` 工具描述，也不是 JSON 字符串**。

课件此处把 Schema 写成 `location / units`，函数却使用 `city / unit`。
下面统一为 `city / unit`，否则传参执行时会报 `unexpected keyword argument 'location'`。

In [17]:
weather_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string", "description": "城市名称"},
        "unit": {"type": "string", "description": "气温单位"},
        "include_forecast": {"type": "boolean"},
    },
    "required": ["city", "unit", "include_forecast"],
}

@tool(args_schema=weather_schema)
def get_weather(
    city: str,
    unit: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """获取当日天气，可选未来五日天气预报。"""
    temp = 22 if unit == "celsius" else 72
    result = f'{city}当日气温：{temp} {"摄氏度" if unit == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

rprint(convert_to_openai_tool(get_weather))
print(get_weather.invoke({
    "city": "北京",
    "unit": "celsius",
    "include_forecast": True,
}))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选未来五日天气预报。',
        'parameters': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string', 'description': '城市名称'},
                'unit': {'type': 'string', 'description': '气温单位'},
                'include_forecast': {'type': 'boolean'}
            },
            'required': ['city', 'unit', 'include_forecast']
        }
    }
}

北京当日气温：22 摄氏度
未来五天都是晴天


这个字典把三个参数都列在 `required` 中，模型看到的就是三个必填参数；
若要描述可选参数，应修改 Schema 本身，不能只依靠 Python 函数的默认值。

**不要把 JSON Schema 字典当作 Pydantic 校验器。** 在当前工程版本中，
字典形式的 `args_schema` 不会像 Pydantic 模型那样自动进行本地参数校验。
需要可靠的本地类型／枚举校验时，优先使用上面的 Pydantic 写法。

### 3.4 几个参数的作用

| 写法 | 作用 |
| --- | --- |
| `@tool` | 把函数转成工具对象 |
| `description="..."` | 显式指定工具用途，覆盖 docstring 的函数说明 |
| `parse_docstring=True` | 解析 Google 风格 Args，生成参数说明 |
| `@tool("getWeather")` | 更改模型看到的工具名 |
| `args_schema=WeatherInput` | 使用 Pydantic 描述参数并在执行前校验 |
| `args_schema=weather_schema` | 使用字典直接提供参数 Schema，适合动态生成描述 |

普通函数的 Schema 转换会解析 docstring 中的参数说明；`@tool` 默认不解析，按需启用。
先看 `convert_to_openai_tool()` 的结果，再直接 `.invoke()` 测试，最后绑定到模型。

参考：[LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)、
[Pydantic 字段](https://docs.pydantic.dev/latest/concepts/fields/)。